# Export the fine-tuned model to GGUF (for Ollama)

Deploy a GPU pod, open Jupyter, and drag in **`sft-adapter.zip`** (the model you downloaded).
Run every cell. At the end, download the `.gguf` file to your PC.

## 1. Install Unsloth

In [ ]:
!pip install -q unsloth

## 2. Unzip the adapter (handles the nested folder)

In [ ]:
import os, zipfile, shutil
if os.path.exists("sft-adapter"): shutil.rmtree("sft-adapter")
with zipfile.ZipFile("sft-adapter.zip") as z: z.extractall("sft-adapter")
ADAPTER_PATH = None
for root, _, fs in os.walk("sft-adapter"):
    if "adapter_config.json" in fs: ADAPTER_PATH = root; break
print("ADAPTER_PATH =", ADAPTER_PATH)
assert ADAPTER_PATH, "adapter_config.json not found in sft-adapter.zip"

## 3. Load + export to GGUF (q4_k_m). Takes ~5-10 min (builds llama.cpp).

In [ ]:
from unsloth import FastLanguageModel
model, tok = FastLanguageModel.from_pretrained(ADAPTER_PATH, max_seq_length=4096, load_in_4bit=True)
model.save_pretrained_gguf("fitness-coach-gguf", tok, quantization_method="q4_k_m")
print("done")

## 4. Find the .gguf to download

In [ ]:
import os
for f in os.listdir("fitness-coach-gguf"):
    p = os.path.join("fitness-coach-gguf", f)
    print(f, f"({os.path.getsize(p)/1e9:.2f} GB)" if os.path.isfile(p) else "")
print("\nRight-click the .gguf in the file browser -> Download.")

## Next (on your PC)
1. Install Ollama: https://ollama.com/download
2. Put the `.gguf` next to `backend/Modelfile`, edit its `FROM` line to the real filename.
3. `ollama create fitness-coach -f backend/Modelfile`
4. `ollama run fitness-coach "test"` to verify.
5. Tell Claude it's running — the backend `.env` gets pointed at Ollama.